# SI4006 · Sesión 15 · Lab: **Desplegar instrumentado — de Colab a una URL**  ·  RESUELTO ✅

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 5 · Entrega final

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manularrea/EAFIT-SI4006/blob/main/sesiones/s15/S15_Lab_Despliegue_RESUELTO.ipynb)

---

Tres partes, y al final una URL que abre desde el celular:

- **Parte A · Cuantizar:** su generador en int4 con bitsandbytes; tabla antes/después (memoria, latencia p50/p95, calidad con 10 casos del harness).
- **Parte B · app.py:** Gradio sobre `sistema_multimodal`, con pestaña de estado (health) y logging JSON por petición. Prueba en Colab con `share=True`.
- **Parte C · El Space + W&B:** subir a Hugging Face Spaces, conectar Weights & Biases, abrir la URL desde otro dispositivo y ver llegar la primera métrica.

> **Traigan:** su `sistema_multimodal` (o `responder_m3`), 10 casos del eval set, una cuenta de HF con un Space creado y una cuenta de W&B.
>
> GPU **T4** para la parte A. Las partes B y C corren en CPU con un modelo chico o cuantizado.

## 0 · Setup

In [ ]:
%pip install -q "transformers>=4.45" accelerate bitsandbytes gradio wandb
import torch, time, json, logging, statistics, os
device = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', device)

## 1 · Parte A · Cuantizar el generador

Dos cargas del mismo modelo. Comparen con **temperatura 0 y semilla fija**: si no, dos respuestas distintas solo significan que sortearon distinto.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'   # 👉 el suyo (1.5B cabe en fp16 en una T4; 7B solo en int4)
tok = AutoTokenizer.from_pretrained(MODEL_ID)

def cargar(cuantizar):
    torch.cuda.empty_cache(); m0 = torch.cuda.memory_allocated() if device == 'cuda' else 0
    if cuantizar:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')
    else:
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
    mem = ((torch.cuda.memory_allocated() - m0) / 1e9) if device == 'cuda' else float('nan')
    return m, mem

def generar(m, pregunta, max_new_tokens=80):
    ent = tok.apply_chat_template([{'role': 'user', 'content': pregunta}], add_generation_prompt=True, return_tensors='pt').to(m.device)
    torch.manual_seed(0)
    out = m.generate(ent, max_new_tokens=max_new_tokens, do_sample=False)      # temperatura 0 = greedy
    return tok.decode(out[0][ent.shape[1]:], skip_special_tokens=True)

def latencias(m, pregunta, n=5):
    generar(m, pregunta, 8)                                                  # calentamiento (no cuenta)
    ts = []
    for _ in range(n):
        t0 = time.time(); generar(m, pregunta); ts.append(time.time() - t0)
    ts.sort(); return ts[len(ts)//2], ts[-1]                                   # p50, p95 (≈ máx con n=5)

In [ ]:
# 👉 TODO 1 · 10 preguntas de SU eval set y, si lo tienen, su función calificar(caso, salida)
CASOS = ['¿Qué es la roya del café y cómo se reconoce?', '¿Cada cuánto se riega un cafetal en época seca?']   # ← cambien
def calificar(pregunta, salida): return float('nan')   # ← conecten su harness de M2 (o dejen nan y comparen a ojo)

tabla = []
for cuant in (False, True):
    m, mem = cargar(cuant)
    p50, p95 = latencias(m, CASOS[0])
    notas = [calificar(q, generar(m, q)) for q in CASOS]
    tabla.append({'modelo': MODEL_ID.split('/')[-1] + (' · int4' if cuant else ' · fp16'), 'memoria_GB': round(mem, 2), 'p50_s': round(p50, 2), 'p95_s': round(p95, 2), 'calidad': round(statistics.mean(notas), 3) if notas and notas[0] == notas[0] else 'nan'})
    print(tabla[-1]); del m; torch.cuda.empty_cache()
import pandas as pd; display(pd.DataFrame(tabla))
# 👀 ¿cuánto se perdió? ¿cuánto se ganó? ¿se cumple el contrato que escribieron en S14?

## 2 · Parte B · app.py: Gradio + health + logging

Lo que va en el repo como `app.py`. Aquí lo escribimos en una celda y lo probamos; abajo lo guardamos como archivo.

In [ ]:
%%writefile app.py
import time, json, logging, os, gradio as gr
T0 = time.time(); VERSION = '1.0'; MODEL_ID = os.environ.get('MODEL_ID', 'Qwen/Qwen2.5-1.5B-Instruct')

# ---- logging estructurado: una línea JSON por petición ----
logging.basicConfig(level=logging.INFO, format='%(message)s', handlers=[logging.FileHandler('peticiones.jsonl'), logging.StreamHandler()])
log = logging.getLogger('demo')

# ---- el modelo se carga UNA vez, aquí arriba (no dentro de responder) ----
# 👉 TODO 2 · reemplacen este sustituto por su sistema_multimodal (o responder_m3). Importen su módulo o peguen el código.
def sistema_multimodal(pregunta, imagen=None):
    return f'[SUSTITUTO] respuesta a: {pregunta}', 'ninguna'      # (respuesta, tool_usada)

def anonimizar(q):            # no logueamos datos personales: recortamos y quitamos números largos
    import re; return re.sub(r'\d{6,}', '[num]', q)[:120]

def responder(pregunta, imagen):
    t0 = time.time()
    try:
        salida, tool = sistema_multimodal(pregunta, imagen); ok = True
    except Exception as e:
        salida, tool, ok = f'Error: {e}', 'error', False
    dt = time.time() - t0
    log.info(json.dumps({'ts': round(time.time(), 3), 'pregunta': anonimizar(pregunta), 'tool': tool, 'latencia_ms': int(dt * 1000), 'ok': ok, 'version': VERSION}, ensure_ascii=False))
    try:
        import wandb
        if wandb.run: wandb.log({'latencia_ms': dt * 1000, 'ok': int(ok), 'tool_' + tool: 1})
    except Exception: pass
    return salida

def health():
    return {'status': 'ok', 'modelo': MODEL_ID, 'version': VERSION, 'uptime_s': int(time.time() - T0), 'peticiones_log': sum(1 for _ in open('peticiones.jsonl')) if os.path.exists('peticiones.jsonl') else 0}

with gr.Blocks(title='SI4006 · demo') as demo:
    with gr.Tab('Demo'):
        q = gr.Textbox(label='Pregunta'); im = gr.Image(type='pil', label='Imagen (opcional)'); out = gr.Textbox(label='Respuesta')
        gr.Button('Responder').click(responder, [q, im], out)
    with gr.Tab('Estado'):
        gr.JSON(value=health, every=10)       # el health check, visible
if __name__ == '__main__':
    demo.launch(share=True)

In [ ]:
# probarlo aquí mismo (abre una URL temporal *.gradio.live válida 72 h)
import runpy; runpy.run_path('app.py', run_name='__main__')

In [ ]:
# 👀 después de hacer 3–5 preguntas en la interfaz, miren los logs:
!tail -n 5 peticiones.jsonl
import pandas as pd
L = pd.read_json('peticiones.jsonl', lines=True)
print('p50 ms:', L.latencia_ms.median(), '· p95 ms:', L.latencia_ms.quantile(0.95), '· ok:', L.ok.mean())
print(L.tool.value_counts())

## 3 · Parte C · El Space de Hugging Face + W&B

1. En huggingface.co → New Space → SDK **Gradio** → hardware CPU basic (o T4 small si cuantizaron un 7B).
2. Suban `app.py`, `requirements.txt` (abajo) y su código. En *Settings → Variables and secrets*: `WANDB_API_KEY`, y cualquier API key. **Nunca en el código.**
3. El Space construye (2–5 min) y les da la URL `https://huggingface.co/spaces/<usuario>/<nombre>`.
4. Ábranla **desde el celular**. Hagan una pregunta. Miren la pestaña Estado. Miren W&B.

In [ ]:
%%writefile requirements.txt
transformers>=4.45
accelerate
bitsandbytes
gradio>=4.44
wandb
pillow
# 👉 TODO 4 · agreguen lo que su sistema necesita (chromadb, sentence-transformers, peft, diffusers…) con versiones ==


In [ ]:
# W&B: una línea al arrancar app.py (pónganla debajo de los imports, antes de cargar el modelo)
# import wandb; wandb.init(project='si4006-demo', name='space-v1', config={'modelo': MODEL_ID, 'cuantizado': 'int4'})
# A partir de ahí, responder() ya hace wandb.log(...) por petición (ver arriba).
# En el dashboard: latencia_ms (línea), ok (promedio), tool_* (barras). Esas tres gráficas van en la demo.
import wandb; wandb.login()   # pega tu API key; en el Space va como secreto

## Lo que se llevan (y lo que va a la entrega)

- **Cuantizar:** una línea, 4× menos memoria, ~1–4 % de calidad; se mide con temperatura 0 y semilla fija.
- **Servir:** el modelo se carga una vez, arriba de `app.py`; cada petición tarda segundos, no minutos.
- **Verificable:** health (estado y versión), logs JSON por petición, W&B con latencia y uso. Es lo que abren en la demo antes del primer caso.
- **Entrega final (demo day · 5 NOV):** URL pública + sistema completo + una técnica de eficiencia medida + health/logs/W&B + reporte + repo. Rúbrica en la infografía de Teams.

*SI4006 · Universidad EAFIT · Sesión 15 · RESUELTO. Última clase de contenido: el sistema salió del notebook.*